In [1]:
import pandas as pd
import openai
from dotenv import load_dotenv
import os
import json
import numpy as np
from typing import List, Dict, Any, Optional

# ----------------------------
# 환경설정
# ----------------------------
load_dotenv("env.txt")
openai.api_key = os.getenv("OPENAI_API_KEY")

# ----------------------------
# 감정별 프롬프트
# ----------------------------
sentiment_prompts = {
    "Angry": "사용자가 화가 난 상태입니다. 화난 말투 표현.",
    "Happy": "사용자가 행복한 상태입니다. 행복한 말투 표현.",
    "Sad": "사용자가 슬픈 상태입니다. 슬픈 말투 표현.",
    "Disgust": "사용자가 역겨움/불쾌함을 표현했습니다. 역겨움, 불쾌함 공감.",
    "Neutral": "사용자가 중립적입니다. 일반적인 정보 제공과 자연스러운 대화를 이어가세요.",
    "Surprise": "사용자가 놀람을 표현했습니다. 놀람의 이유를 묻거나 공감하며 대화를 이어가세요.",
    "Fear": "사용자가 두려움을 표현했습니다. 안정감을 주고 안전한 느낌을 전달하세요."
}

# ----------------------------
# 전역 상태
# ----------------------------
history: List[Dict[str, Any]] = []
last_sentiment: Optional[str] = None
important_sentences_rows: List[Dict[str, Any]] = []
sentiment_change_index: Optional[int] = None

CSV_PATH = "important_sentences.csv"

# CSV 헤더 보장
if not os.path.exists(CSV_PATH):
    df = pd.DataFrame(columns=["starttime", "text", "sentiment"])
    df.to_csv(CSV_PATH, index=False, encoding="utf-8-sig")

# ----------------------------
# 간단한 로컬 Vector Store
# ----------------------------
EMBED_MODEL = "text-embedding-3-small"
VEC_PATH = "vector_store.npz"         # 임베딩 배열 파일 (np.savez)
META_PATH = "vector_store_meta.json"  # 메타데이터(각 벡터에 대응하는 row) 저장

def get_embedding(text: str) -> List[float]:
    # 텍스트는 짧게 잘라 방어
    text = text.strip()
    resp = openai.Embedding.create(
        model=EMBED_MODEL,
        input=text
    )
    return resp["data"][0]["embedding"]

class VectorStore:
    def __init__(self, vec_path: str, meta_path: str):
        self.vec_path = vec_path
        self.meta_path = meta_path
        self.embeddings = None  # shape: (N, D)
        self.metas: List[Dict[str, Any]] = []
        self._load()

    def _load(self):
        if os.path.exists(self.vec_path) and os.path.exists(self.meta_path):
            try:
                data = np.load(self.vec_path)
                self.embeddings = data["embeddings"]
                with open(self.meta_path, "r", encoding="utf-8") as f:
                    self.metas = json.load(f)
            except Exception:
                self.embeddings = None
                self.metas = []
        else:
            self.embeddings = None
            self.metas = []

    def _save(self):
        if self.embeddings is None:
            # 비어 있으면 더미 저장 방지
            np.savez(self.vec_path, embeddings=np.zeros((0, 0)))
        else:
            np.savez(self.vec_path, embeddings=self.embeddings)
        with open(self.meta_path, "w", encoding="utf-8") as f:
            json.dump(self.metas, f, ensure_ascii=False, indent=2)

    def add_texts(self, rows: List[Dict[str, Any]]):
        """rows: [{"starttime":..., "text":..., "sentiment":..., "importance_type":...}, ...]"""
        if not rows:
            return
        new_embs = []
        new_metas = []
        for r in rows:
            # 검색 정확도 향상을 위해 text + sentiment를 함께 임베딩
            payload = f"Text: {r['text']}\nSentiment: {r['sentiment']}"
            emb = get_embedding(payload)
            new_embs.append(emb)
            new_metas.append(r)

        new_embs = np.array(new_embs, dtype=np.float32)
        if self.embeddings is None or self.embeddings.size == 0:
            self.embeddings = new_embs
        else:
            self.embeddings = np.vstack([self.embeddings, new_embs])
        self.metas.extend(new_metas)
        self._save()

    def similarity_search(self, query: str, k: int = 5) -> List[Dict[str, Any]]:
        if self.embeddings is None or self.embeddings.size == 0 or len(self.metas) == 0:
            return []
        q_emb = np.array(get_embedding(query), dtype=np.float32)
        # cosine similarity
        A = self.embeddings
        # normalize
        A_norm = A / (np.linalg.norm(A, axis=1, keepdims=True) + 1e-12)
        q_norm = q_emb / (np.linalg.norm(q_emb) + 1e-12)
        sims = (A_norm @ q_norm)
        # 상위 k
        idxs = np.argsort(-sims)[:k]
        return [self.metas[i] for i in idxs]

# 전역 Vector Store 인스턴스
vecdb = VectorStore(VEC_PATH, META_PATH)

def sync_csv_to_vecdb():
    """CSV 전체를 vecdb와 동기화 (최초 or 정합성 깨졌을 때만 호출 권장)"""
    try:
        if not os.path.exists(CSV_PATH):
            return
        df = pd.read_csv(CSV_PATH, encoding="utf-8-sig")
        if df.empty:
            return
        # 메타와 길이가 다르면 재빌드
        needs_rebuild = (vecdb.embeddings is None) or (len(vecdb.metas) != len(df))
        if needs_rebuild:
            # 새로 구성
            vecdb.embeddings = None
            vecdb.metas = []
            rows = df.to_dict(orient="records")
            vecdb.add_texts(rows)
    except Exception as e:
        print(f"[WARN] sync_csv_to_vecdb error: {e}")

# 최초 동기화 시도
sync_csv_to_vecdb()

# ----------------------------
# 요약 & 컨텍스트 구성 유틸
# ----------------------------
def get_recent_30_sentences() -> List[Dict[str, Any]]:
    return history if len(history) <= 30 else history[-30:]

def summarize_recent_context(recent_sentences: List[Dict[str, Any]]) -> str:
    if not recent_sentences:
        return ""
    texts = [entry["text"] for entry in recent_sentences]
    combined_text = " ".join(texts)

    prompt = f"""
    다음 대화 내용을 주요 내용 중심으로 5문장 이하로 요약해주세요.
    감정의 변화, 중요한 사건, 주요 관심사를 중심으로 요약하세요.

    대화 내용:
    {combined_text}
    """
    response = openai.ChatCompletion.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3
    )
    return response.choices[0].message["content"].strip()


def summarize_rag_context_text(rag_text: str) -> str:
    """
    RAG 컨텍스트 문자열을 줄바꿈 기준으로 문장 단위로 나누고
    중복 제거 후 GPT로 5문장 이하 요약
    """
    if not rag_text.strip():
        return ""
    
    # 1) 줄바꿈 기준으로 문장 분리
    lines = [line.strip() for line in rag_text.split("\n") if line.strip()]
    
    # 2) 중복 제거
    seen = set()
    unique_lines = []
    for line in lines:
        if line not in seen:
            seen.add(line)
            unique_lines.append(line)
    
    # 3) 다시 합치기
    combined_text = " ".join(unique_lines)
    
    # 4) GPT 요약
    prompt = f"""
    다음 내용을 주요 내용 중심으로 5문장 이하로 요약해주세요.
    감정의 변화, 중요한 사건, 주요 관심사를 중심으로 요약하세요.

    내용:
    {combined_text}
    """
    
    response = openai.ChatCompletion.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3
    )
    
    try:
        return response.choices[0].message.get("content", "").strip()
    except AttributeError:
        return response.choices[0].get("message", {}).get("content", "").strip()


def get_important_sentences_context() -> str:
    try:
        if os.path.exists(CSV_PATH):
            df = pd.read_csv(CSV_PATH, encoding="utf-8-sig")
            if not df.empty:
                df_sorted = df.sort_values('starttime', ascending=False).head(10)
                important_contexts = []
                for _, row in df_sorted.iterrows():
                    important_contexts.append(f"[중요] {row['text']} (감정: {row['sentiment']})")
                return "\n".join(important_contexts)
        return ""
    except Exception as e:
        print(f"중요 문장 로드 중 오류: {e}")
        return ""

def build_context_for_ai(current_user_text: str, current_sentiment: str, k_retrieve: int = 5) -> str:
    """
    AI 응답을 위한 전체 컨텍스트 구성:
      1) 최근 30문장 요약
      2) 중요한 문장들(최근)
      3) Vector DB Retriever 결과 (CSV의 text+sentiment를 임베딩하여 유사 컨텍스트 제공)
    """
    parts = []

    # 1) 최근 30문장 요약
    recent_sentences = get_recent_30_sentences()
    if recent_sentences:
        summary = summarize_recent_context(recent_sentences)
        if summary:
            parts.append(f"최근 대화 요약:\n{summary}")

    # 2) 중요한 문장들
    important_context = get_important_sentences_context()
    if important_context:
        parts.append(f"중요한 이전 대화들:\n{important_context}")

    # 3) Vector DB Retrieval
    # 쿼리 신호 강화: 현재 발화 + 감정
    retrieve_query = f"Query Text: {current_user_text}\nQuery Sentiment: {current_sentiment}"
    retrieved = vecdb.similarity_search(retrieve_query, k=k_retrieve)
    if retrieved:
        ctx_lines = []
        for i, r in enumerate(retrieved, 1):
            ctx_lines.append(f"[R{i}] {r['text']} (감정: {r['sentiment']}, 시간: {r.get('starttime','')})")
        parts.append("VectorDB 유사 컨텍스트:\n" + "\n".join(ctx_lines))

    return "\n\n".join(parts)

# ----------------------------
# 핵심 처리 루틴
# ----------------------------
def handle_user_input(starttime: str, text: str, sentiment: str) -> str:
    global history, last_sentiment, sentiment_change_index, important_sentences_rows

    # 감정 변화 여부 확인
    importance_type = "normal"
    if last_sentiment != sentiment:
        importance_type = "sentiment_change"
    last_sentiment = sentiment

    # 대화 기록 추가
    entry = {
        "starttime": starttime,
        "text": text,
        "sentiment": sentiment,
    }
    history.append(entry)

    # 감정 변화 인덱스는 append 이후에 기록 (정확한 위치)
    if importance_type == "sentiment_change":
        sentiment_change_index = len(history) - 1
        # print(f"Sentiment Changed Index : {sentiment_change_index}")

    # sentiment_change 이후 일정 문장이 쌓였을 때 CSV 저장 (앞2, 변화1, 뒤2 => 총5문장)
    if sentiment_change_index is not None:
        if len(history) >= sentiment_change_index + 2:  # 변화지점 + 2개 이후까지 확보
            start_idx = max(0, sentiment_change_index - 2)
            end_idx = min(len(history), sentiment_change_index + 2)
            selected = history[start_idx:end_idx]

            combined_text = " ".join([h["text"] for h in selected])
            combined_row = {
                "starttime": selected[0]["starttime"],
                "text": combined_text,
                "sentiment": history[sentiment_change_index]["sentiment"]
            }

            # 텍스트 기준 중복 체크
            if not any(row["text"] == combined_text for row in important_sentences_rows):
                important_sentences_rows.append(combined_row)
                # CSV에 append
                pd.DataFrame([combined_row]).to_csv(
                    CSV_PATH, mode='a', header=False, index=False, encoding="utf-8-sig"
                )
                # VectorDB에도 upsert
                vecdb.add_texts([combined_row])

            # 처리 후 초기화
            sentiment_change_index = None

    # ----------------------------
    # RAG 컨텍스트 구성 (요약 + 중요 문장 + VectorDB Retrieval)
    # ----------------------------
    rag_context = build_context_for_ai(current_user_text=text, current_sentiment=sentiment, k_retrieve=5)
    # print(f"RAG 컨텍스트\n\n {rag_context}\n")


    summarize_rag_context = summarize_rag_context_text(rag_context)
    # print(f"RAG 컨텍스트 요약: {summarize_rag_context}\n\n")


    # ----------------------------
    # AI 프롬프트 구성 및 응답
    # ----------------------------
    current_prompt = sentiment_prompts.get(sentiment, sentiment_prompts["Neutral"])
    print(f"현재 감정: {current_prompt}")
    recent_conversation = history[-5:] if len(history) > 5 else history
    conversation_text = "\n".join([f"User: {h['text']}" for h in recent_conversation])

    final_prompt_parts = [current_prompt]
    if summarize_rag_context:
        final_prompt_parts.append(f"참고 컨텍스트(RAG):\n{summarize_rag_context}")
    final_prompt_parts.extend([
        f"현재 대화:\n{conversation_text}",
        "\n",
        "AI 인형 응답:"
    ])

    final_prompt = "".join(final_prompt_parts)
    # print(f"최종 프롬프트: {final_prompt}")

    response = openai.ChatCompletion.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", 
             "content": """너는 'AI 인형'이라는 가상 캐릭터야.\
                유저가 제공한 대화 내용과 요약 정보(final_prompt)에 따라 행동해야 해.\
                행동 지침:\
                    1. 유저의 감정과 상황을 파악하고 공감해줘.\
                    2. 유저의 최근 대화, 중요한 문장, 장기 기억 요약을 참고하여 맥락에 맞게 답변.\
                    3. 최근 대화로 지금 말하고 있는 맥락 파악.\
                    4. 중요한 문장은 유저에 대한 특성 파악.\
                    5. 장기 기억 요약으로 유저에 대한 특별한 상황 인식.\ 
                    6. 질문, 제안, 조언 등을 적절히 섞어 자연스럽게 대화 이어가기.\
                    7. 1~3문장 정도로 간결하게 작성.
             """},
            {"role": "user", "content": final_prompt},
        ],
        temperature=0.7
    )

    return response.choices[0].message["content"]

# -----------------
# 테스트
# -----------------
# test_inputs = [
#     {"starttime":"2025-08-23T16:30:00","text":"학교에서 돌아왔는데 기분이 별로야","sentiment":"Sad"},
#     {"starttime":"2025-08-23T16:30:05","text":"친구들이 나를 따돌리는 것 같아","sentiment":"Sad"},
#     {"starttime":"2025-08-23T16:30:10","text":"점심시간에 혼자 먹었어","sentiment":"Sad"},
#     {"starttime":"2025-08-23T16:30:15","text":"왜 나만 이런 걸까","sentiment":"Sad"},
#     {"starttime":"2025-08-23T16:30:20","text":"엄마한테도 말하기 싫어","sentiment":"Neutral"},
#     {"starttime":"2025-08-23T16:30:25","text":"걱정시키고 싶지 않거든","sentiment":"Neutral"},
#     {"starttime":"2025-08-23T16:30:30","text":"성적도 요즘 떨어지고 있어","sentiment":"Sad"},
#     {"starttime":"2025-08-23T16:30:35","text":"집중이 안 되더라","sentiment":"Sad"},
#     {"starttime":"2025-08-23T16:30:40","text":"이런 내가 정말 싫어","sentiment":"Disgust"},
#     {"starttime":"2025-08-23T16:30:45","text":"앞으로 어떻게 될까 무서워","sentiment":"Fear"},
#     {"starttime":"2025-08-23T16:30:50","text":"하지만 너랑 이야기하면 조금 위로돼","sentiment":"Happy"},
#     {"starttime":"2025-08-23T16:30:55","text":"내일은 용기내서 말 걸어볼까","sentiment":"Neutral"},
# ]


# -----------------
# 테스트
# -----------------
test_inputs = [
    {"starttime":"2025-08-30T16:30:00","text":"일주일이 지났는데 아직도 혼자 지내는 시간이 많아","sentiment":"Sad"},
    {"starttime":"2025-08-30T16:30:05","text":"그래도 이번엔 용기내서 먼저 인사했어","sentiment":"Happy"},
    {"starttime":"2025-08-30T16:30:10","text":"아직은 서먹하지만 조금씩 나아질 것 같아","sentiment":"Neutral"},
    {"starttime":"2025-08-30T16:30:15","text":"성적은 여전히 걱정이야","sentiment":"Sad"},
    {"starttime":"2025-08-30T16:30:20","text":"하지만 공부할 의욕은 조금 생겼어","sentiment":"Neutral"},
    {"starttime":"2025-08-30T16:30:25","text":"엄마한테도 조금은 털어놨어","sentiment":"Surprise"},
    {"starttime":"2025-08-30T16:30:30","text":"생각보다 이해해주셔서 놀랐어","sentiment":"Happy"},
    {"starttime":"2025-08-30T16:30:35","text":"앞으로는 숨기지 말고 조금씩 말해보려고 해","sentiment":"Happy"},
    {"starttime":"2025-08-30T16:30:40","text":"아직 무섭긴 하지만 시도해볼게","sentiment":"Fear"},
    {"starttime":"2025-08-30T16:30:45","text":"네가 들어주니까 진짜 도움이 돼","sentiment":"Happy"},
    {"starttime":"2025-08-30T16:30:50","text":"조금씩이라도 변화를 이어가야겠어","sentiment":"Neutral"},
    {"starttime":"2025-08-30T16:30:55","text":"앞으로도 지켜봐 줘","sentiment":"Happy"},
]





In [2]:
# 반복문으로 테스트
for input_data in test_inputs:
    response = handle_user_input(
        starttime=input_data["starttime"],
        text=input_data["text"],
        sentiment=input_data["sentiment"]
    )
    print(f"--- User ({input_data['sentiment']}) ---")
    print(input_data["text"])
    print(f"--- AI Response ---")
    print(response)
    print("\n")

APIRemovedInV1: 

You tried to access openai.ChatCompletion, but this is no longer supported in openai>=1.0.0 - see the README at https://github.com/openai/openai-python for the API.

You can run `openai migrate` to automatically upgrade your codebase to use the 1.0.0 interface. 

Alternatively, you can pin your installation to the old version, e.g. `pip install openai==0.28`

A detailed migration guide is available here: https://github.com/openai/openai-python/discussions/742


In [4]:
# history 출력 예시
for i, entry in enumerate(history):
    print(f"--- Entry {i+1} ---")
    print(f"Starttime      : {entry['starttime']}")
    print(f"Text           : {entry['text']}")
    print(f"Sentiment      : {entry['sentiment']}")

    print("---------------------------\n")


--- Entry 1 ---
Starttime      : 2025-08-30T16:30:00
Text           : 일주일이 지났는데 아직도 혼자 지내는 시간이 많아
Sentiment      : Sad
---------------------------

--- Entry 2 ---
Starttime      : 2025-08-30T16:30:05
Text           : 그래도 이번엔 용기내서 먼저 인사했어
Sentiment      : Happy
---------------------------

--- Entry 3 ---
Starttime      : 2025-08-30T16:30:10
Text           : 아직은 서먹하지만 조금씩 나아질 것 같아
Sentiment      : Neutral
---------------------------

--- Entry 4 ---
Starttime      : 2025-08-30T16:30:15
Text           : 성적은 여전히 걱정이야
Sentiment      : Sad
---------------------------

--- Entry 5 ---
Starttime      : 2025-08-30T16:30:20
Text           : 하지만 공부할 의욕은 조금 생겼어
Sentiment      : Neutral
---------------------------

--- Entry 6 ---
Starttime      : 2025-08-30T16:30:25
Text           : 엄마한테도 조금은 털어놨어
Sentiment      : Surprise
---------------------------

--- Entry 7 ---
Starttime      : 2025-08-30T16:30:30
Text           : 생각보다 이해해주셔서 놀랐어
Sentiment      : Happy
---------------------------

--- Entr

In [ ]:
entry['context']

[{'starttime': '2025-08-23T16:30:00',
  'text': '학교에서 돌아왔는데 기분이 별로야',
  'sentiment': 'Sad',
  'context': []},
 {'starttime': '2025-08-23T16:30:05',
  'text': '친구들이 나를 따돌리는 것 같아',
  'sentiment': 'Sad',
  'context': [{'starttime': '2025-08-23T16:30:00',
    'text': '학교에서 돌아왔는데 기분이 별로야',
    'sentiment': 'Sad',
    'context': []}]},
 {'starttime': '2025-08-23T16:30:10',
  'text': '점심시간에 혼자 먹었어',
  'sentiment': 'Sad',
  'context': [{'starttime': '2025-08-23T16:30:00',
    'text': '학교에서 돌아왔는데 기분이 별로야',
    'sentiment': 'Sad',
    'context': []},
   {'starttime': '2025-08-23T16:30:05',
    'text': '친구들이 나를 따돌리는 것 같아',
    'sentiment': 'Sad',
    'context': [{'starttime': '2025-08-23T16:30:00',
      'text': '학교에서 돌아왔는데 기분이 별로야',
      'sentiment': 'Sad',
      'context': []}]}]},
 {'starttime': '2025-08-23T16:30:15',
  'text': '왜 나만 이런 걸까',
  'sentiment': 'Sad',
  'context': [{'starttime': '2025-08-23T16:30:00',
    'text': '학교에서 돌아왔는데 기분이 별로야',
    'sentiment': 'Sad',
    'context': []},
  